# 2-4. Advanced RAG: Chunking 개선

## 학습 목표

- 청크 크기에 따라 검색 결과가 어떻게 달라지는지 비교한다.
- PDF 매뉴얼에 적합한 청크 크기를 실험한다.

## 공통 전제

- 실습 데이터: `../data/공직자_민원응대_핵심_매뉴얼.pdf`
- 기본 모델: `gpt-4o-mini`
- 기본 임베딩 모델: `text-embedding-3-small`
- 벡터DB: Qdrant 로컬 인메모리 모드

> 이 노트북은 `src` 파일을 import하지 않고, 노트북 안의 코드만으로 실행되도록 구성한다.

In [ ]:
from pathlib import Path
import re
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

load_dotenv(override=True, dotenv_path="../../.env")

PDF_PATH = Path("../data/공직자_민원응대_핵심_매뉴얼.pdf")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
def clean_text(text: str) -> str:
    """PDF에서 추출한 텍스트를 기본 정제한다."""
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def load_pdf_pages(pdf_path: Path) -> list[Document]:
    """PDF를 페이지 단위 Document 리스트로 로드한다."""
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()
    page_docs = []

    for doc in docs:
        text = clean_text(doc.page_content or "")
        if not text:
            continue

        page_docs.append(
            Document(
                page_content=text,
                metadata={
                    **doc.metadata,
                    "source": pdf_path.name,
                    "page": doc.metadata.get("page", 0) + 1, 
                }
            )
        )
    return page_docs


def make_chunks(docs: list[Document], chunk_size: int, chunk_overlap: int) -> list[Document]:
    """페이지 단위 문서를 검색용 청크로 분할한다."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", "STEP", "‣", "•", ". ", " "]
    )
    chunks = splitter.split_documents(docs)

    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i
        chunk.metadata["chunk_size"] = chunk_size

    return chunks

## 1. 서로 다른 청크 크기 만들기

In [ ]:
pages = load_pdf_pages(PDF_PATH)

small_chunks = make_chunks(pages, chunk_size=400, chunk_overlap=80)
medium_chunks = make_chunks(pages, chunk_size=700, chunk_overlap=120)
large_chunks = make_chunks(pages, chunk_size=1000, chunk_overlap=150)

print("small:", len(small_chunks))
print("medium:", len(medium_chunks))
print("large:", len(large_chunks))

small: 70
medium: 43
large: 32


## 2. 청크 크기별 벡터스토어 생성

In [4]:
small_vs = QdrantVectorStore.from_documents(
    documents=small_chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="small_chunks",
    force_recreate=True
)

medium_vs = QdrantVectorStore.from_documents(
    documents=medium_chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="medium_chunks",
    force_recreate=True
)

large_vs = QdrantVectorStore.from_documents(
    documents=large_chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="large_chunks",
    force_recreate=True
)

## 3. 검색 결과 비교

In [5]:
question = "전화 중 욕설을 하면 어떻게 대응해야 하나요?"

stores = {
    "small": small_vs,
    "medium": medium_vs,
    "large": large_vs
}

for name, store in stores.items():
    print("\n" + "=" * 80)
    print(f"[{name} chunk 검색 결과]")
    docs = store.similarity_search(question, k=3)

    for doc in docs:
        print(doc.metadata)
        print(doc.page_content[:350].replace("\n", " "))
        print()


[small chunk 검색 결과]
{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 16, 'chunk_id': 46, 'chunk_size': 400, '_id': '8bb8f94153e2415dade362b15fd1e27d', '_collection_name': 'small_chunks'}
• 인·허가 등 정상적인 업무처리 과정에서 민원인과의 상담이 길어지는 경우는 정당한 사유없는 장시간   전화에 해당하지 않으므로 종결대상에 해당하지 않음. 질의 3 상급자 응대 후에도 계속 전화해서 폭언을 하는 경우는 어떻게 대응 해야 하는지? • 담당자에게 휴게시간 부여 등 응대를 중지하도록 조치하고 대행자 등은 다른 민원처리를 위해 해당   민원은 일시적으로 응대 보류함. • 폭언, 업무방해 등에 대해 경고문 발송, 고소·고발 등 법적 대응을 추진하여 위법행위를 지속하지   않도록 강력 대응.

{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 8, 'chunk_id': 14, 'chunk_size': 400, '_id': '7731440504cf4b6299254c04bb395fd6', '_collection_name': 'small_chunks'}
반복전화를 받은 부서원에게 30분 이내 휴게시간 부여 가능  ※ 통화 내용과 정도를 감안하여 필요시 위법행위 관리대장을 작성하여 채증 2단계 상황 보고 핵심대응 다른민원 처리를 위해 통화 지속 곤란 안내 후 종료 ‣ 통화 지속 곤란을 설명한 후 상담종료   정당한 행정처분에 불복하여 반복전화하는 경우  선생님, 말씀하신 민원은 정해진 규정과 절차에 따라 정상적으로 처리된   사항입니다. (또는 현재 처리 중에 있습니다.) 동일한 답변 외에는 도움을   드릴 수 없으므로 다른 민원처리를 위해 통화를 종료하오니 양해하여 주시기   바랍니다. ※ 국민권익위원회 고충민원 등 신청 안내  충분한 설명이 이루어졌음에도 반복전화하는

{'source': '공직자_민원응대_핵심_매뉴

“작은 청크는 잘 찾지만 너무 잘게 쪼개져서 답변에 필요한 절차가 끊길 수 있고, 큰 청크는 문맥은 많지만 관련 없는 내용까지 섞인다. 이 실험에서는 중간 크기 청크가 검색 정확도와 답변 생성에 필요한 문맥을 가장 균형 있게 제공한다.”

# chunk 선택 판단

| 구분     |                 설정 | 청크 수 | 판단                           |
| ------ | -----------------: | ---: | ---------------------------- |
| small  |   400 / overlap 80 |  70개 | 검색 정확도는 좋지만 문맥이 짧게 잘릴 위험이 있다 |
| medium |  700 / overlap 120 |  43개 | 검색 정확도와 문맥 보존의 균형이 가장 좋다     |
| large  | 1000 / overlap 150 |  32개 | 문맥은 풍부하지만 불필요한 내용이 함께 섞인다    |


## 청크 수 판단 기준
| 평가 기준      | small | medium | large    |
| ---------- | ----- | ------ | -------- |
| 검색 정확도     | 높음    | 높음     | 중간       |
| 문맥 보존      | 낮음    | 높음     | 매우 높음    |
| 불필요한 정보 포함 | 낮음    | 적절함    | 높음       |
| 답변 생성 안정성  | 보통    | 가장 좋음  | 보통       |
| 수업 실습 적합성  | 좋음    | 가장 좋음  | 설명용으로 좋음 |


## 핵심 정리

- 작은 청크는 정확한 구간을 찾기 좋지만 문맥이 부족할 수 있다.
- 큰 청크는 문맥이 풍부하지만 불필요한 내용이 섞일 수 있다.
- 매뉴얼형 문서는 제목, 단계, 항목을 고려한 청크 분할이 중요하다.